# FinOps Budget Routing Control Plane — epic proof

Runs the offline routing simulation from `services/finops-routing-control-plane.ts` (same module as the cucumber steps): a provider-neutral dry run emits a DecisionGraph and backend-neutral ActionPlan, and governed user metrics that touch raw telemetry are rejected before routing begins.

In [1]:
import { strict as assert } from "node:assert";

import {
	parseMetricContracts,
	runFinopsRoutingControlPlaneSimulation,
} from "./services/finops-routing-control-plane.ts";

const sim = await runFinopsRoutingControlPlaneSimulation();
const route = sim.approvedRoute;

// 1) Provider-neutral dry run: DecisionGraph + backend-neutral ActionPlan.
assert.equal(route.parse.intent.ok, true);
assert.ok(route.decisionGraph.id);
assert.ok(route.actionPlan);
assert.doesNotMatch(
	JSON.stringify(route.actionPlan),
	/terraform|pulumi|crossplane|kubernetes/iu,
);
console.log(
	`dry run intentParsed=${route.parse.intent.ok} decisionGraph=${route.decisionGraph.id} actionPlanNeutral=true`,
);

// 2) Governed metrics: user metrics referencing raw telemetry fail before routing.
const metricParse = parseMetricContracts([
	{
		name: "team-preference",
		tier: "user",
		version: "v1.0.0",
		requiresApproval: true,
		compositionSources: ["raw:otel.cpu.seconds"],
	},
]);
assert.equal(metricParse.ok, false);
assert.ok(
	metricParse.errors.some((error: string) => error.includes("raw telemetry")),
);
assert.deepEqual(metricParse.contracts, []);
console.log(
	`metric governance ok=${metricParse.ok} errors=${metricParse.errors.length} contractsReachingRouting=${metricParse.contracts.length}`,
);

dry run intentParsed=true decisionGraph=decision-graph:approved-route actionPlanNeutral=true
metric governance ok=false errors=1 contractsReachingRouting=0


Routing discipline: constraints filter before Pareto ranking (rejected candidates are never scored), preference weights select only inside the frontier, sub-threshold improvements are damped by the stability controller, and feedback lands as gated proposals.

In [2]:
// 3) Constraint-first Pareto routing: rejected candidates are never scored;
//    preference weights select only inside the frontier.
assert.ok(route.constraints.rejected.length > 0);
const vectorIds = new Set(
	route.metricVectors.map(
		(vector: { candidateId: string }) => vector.candidateId,
	),
);
for (const rejected of route.constraints.rejected) {
	assert.equal(vectorIds.has(rejected.id ?? rejected.candidateId), false);
}
assert.ok(route.pareto.candidateIds.includes(route.preference.selectedCandidateId));
console.log(
	`routing rejected=${route.constraints.rejected.length} scored=${route.metricVectors.length} frontier=${route.pareto.candidateIds.length} selected=${route.preference.selectedCandidateId}`,
);

// 4) Stability cooldown: sub-threshold improvements keep the current placement.
const stability = sim.stabilityCooldown.stability;
assert.equal(stability.selectedCandidateId, "candidate-economy");
console.log(`stability cooldown retains=${stability.selectedCandidateId}`);

// 5) Feedback lands as gated proposals, never as direct metric/policy mutation.
assert.ok(sim.feedbackGovernance.proposals.length > 0);
assert.ok(
	sim.feedbackGovernance.proposals.every((proposal: { id: string }) =>
		proposal.id.startsWith("feedback-proposal:"),
	),
);
console.log(
	`feedback proposals=${sim.feedbackGovernance.proposals.length} allGated=true`,
);

console.log(
	"EPIC HELD: routing is constraint-first, provider-neutral, stability-damped, and proposal-gated",
);

routing rejected=1 scored=4 frontier=3 selected=candidate-balanced
stability cooldown retains=candidate-economy
feedback proposals=2 allGated=true
EPIC HELD: routing is constraint-first, provider-neutral, stability-damped, and proposal-gated


Every assertion above ran green — the same invariants the executable `features/*.feature` scenarios enforce in the repo cucumber lane. `examples/usage.ts` is the gate-executed twin of this notebook.